In [18]:
import logging
import os
import threading
from abc import ABC, abstractmethod
import ast
import patito as pt

from pathlib import Path
from typing import Any, Dict, Iterator, Optional, Type
import json

from pyarrow.parquet import ParquetWriter
import httpx
import polars as pl
import pyarrow.parquet as pq
import yaml
import zstandard as zstd
import queue
from dataclasses import dataclass
from typing import Any, Iterable, List, Optional, Tuple

import experiment.main_registry  # important for correctly loading registered classes
from data_connectors.mmlu_pro import MMLUProExample, MMLUProCategory
from experiment.main_registry import DataConnectorName
from concurrent.futures import ThreadPoolExecutor, Future
from pydantic import ConfigDict
from utils.hydra_config import MainConfig
from utils.tracking import TrackEntry
import pyarrow

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
RUNS_DIR = "/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/social_studies/results/runs"
PHOENIX_ENDPOINT = "http://localhost:6006"

In [20]:
def read_hydra_config(run_dir: Path) -> MainConfig:
    """
    Common Hydra output: <run_dir>/.hydra/config.yaml (and overrides.yaml).
    Adjust paths for your setup.
    """
    cfg_path = run_dir / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)

    config = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

    config = MainConfig.model_validate(config, strict=False)

    # TODO: Remove when correctly set:
    if config.meta_info.output_directory == Path("/tmp"):
        config.meta_info.output_directory = run_dir

    return config


def read_log_text(run_dir: Path, log_filename: str = "main.log") -> Optional[str]:
    p = run_dir / log_filename
    if not p.exists():
        return None
    return p.read_text(encoding="utf-8", errors="replace")


def iter_jsonl_zst(path: Path) -> Iterator[Dict[str, Any]]:
    """
    Streams a .jsonl.zst file and yields dict per line.
    """
    with path.open("rb") as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            buf = b""
            while True:
                chunk = reader.read(1 << 20)
                if not chunk:
                    break
                buf += chunk
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    if not line.strip():
                        continue
                    yield json.loads(line)
            if buf.strip():
                yield json.loads(buf)

In [21]:
def get_span_attributes(phoenix_graphql_endpoint: str, *, spanIds: list[str]):
    fields = "\n".join(
        f's_{oid}: getSpanByOtelId(spanId: "{oid}") {{ attributes }}' for oid in spanIds
    )
    query = f"query GetSpans {{\n{fields}\n}}"

    with httpx.Client() as client:
        response = client.post(
            phoenix_graphql_endpoint,
            json={
                "query": query,
            },
            headers={"Content-Type": "application/json"},
        )

    response.raise_for_status()
    data = response.json()

    if "errors" in data:
        raise RuntimeError(data["errors"])

    out = {}
    for alias, node in data["data"].items():
        original_oid = alias[2:]
        if node is None:
            out[original_oid] = None
            continue

        attrs = node["attributes"]
        span = json.loads(attrs) if isinstance(attrs, str) else attrs
        out[original_oid] = span

    return out

In [22]:
class PolarsBaseModel(pt.Model, ABC):
    @classmethod
    @abstractmethod
    def from_raw_data(cls, *args, **kwargs) -> "PolarsBaseModel": ...

    @classmethod
    def get_polars_schema(cls) -> pl.Schema:
        return pl.Schema(cls.examples().schema)

    @classmethod
    def create_parquet_table(cls, items: list["PolarsBaseModel"]):
        schema = cls.get_polars_schema()

        return (
            cls.DataFrame([item.model_dump() for item in items], orient="row", schema=schema)
            .cast(strict=True)
            .validate()
            .to_arrow()
            .cast(polars_schema_to_arrow_schema(schema))
        )

In [23]:
class Question(PolarsBaseModel):
    model_config = ConfigDict(use_enum_values=True)
    question_id: int

    original_question_id: int

    data_connector: DataConnectorName
    original_source: str
    category: str
    question: str
    answer_options: list[str]
    answer_index: int
    answer_string: str

    @classmethod
    def from_raw_data(
            cls,
            *,
            assigned_id: int,
            entry: TrackEntry,
            hydra_config: MainConfig,
            span_attributes: dict[str, Any],
    ) -> "Question":
        match hydra_config.experiment.data.value:
            case "mmlu-pro":
                # TODO: Remove this when new data is generated
                if not isinstance(
                        options := span_attributes["question"]["options"], list
                ):
                    span_attributes["question"]["options"] = ast.literal_eval(options)

                if not isinstance(
                        category := span_attributes["question"]["category"], MMLUProCategory
                ):
                    span_attributes["question"]["category"] = MMLUProCategory[
                        category.split(".")[-1]
                    ]

                question = MMLUProExample.model_validate(span_attributes["question"])
                return cls(
                    question_id=assigned_id,
                    original_question_id=question.question_id,
                    data_connector=hydra_config.experiment.data,
                    original_source=question.src,
                    category=question.category.value,
                    question=entry.input.question,
                    answer_options=question.options,
                    answer_index=question.answer_index,
                    answer_string=question.answer,
                )
            case _:
                raise NotImplementedError("Data Connector not Implemented yet")

In [24]:
MessageId = int


class Answer(PolarsBaseModel):
    id: int
    run_id: int
    message_ids: List[MessageId]
    question_id: int
    phoenix_span_id: str
    run_identifier: str
    answers_at_beginning: List[str]
    answers_at_end: List[str]
    final_answer: Optional[str]

    @classmethod
    def from_raw_data(
            cls,
            *,
            assigned_id: int,
            run_id: int,
            entry: TrackEntry,
            hydra_config: MainConfig,
            question_id: int,
    ) -> "Answer":
        # TODO: Change to hydra_config.meta_info.phoenix_project_name, when correctly set
        phoenix_project_name = (
                hydra_config.experiment.name
                + " - "
                + hydra_config.meta_info.output_directory.name
        )
        return cls(
            id=assigned_id,
            run_id=run_id,
            message_ids=[],  # TODO
            question_id=question_id,
            phoenix_span_id=entry.phoenix_span_info.span_id_hex,
            run_identifier=phoenix_project_name,
            answers_at_beginning=entry.output.answers_at_beginning or [],
            answers_at_end=entry.output.answers_at_end or [],
            final_answer=entry.output.final_answer,
        )

In [25]:
class Run(PolarsBaseModel):
    run_id: int
    experiment_id: int
    execution_config_json: str
    meta_info_json: str

    @classmethod
    def from_raw_data(
            cls, *, assigned_id: int, hydra_config: MainConfig, experiment_id: int
    ) -> "Run":
        return cls(
            run_id=assigned_id,
            experiment_id=experiment_id,
            execution_config_json=hydra_config.execution.model_dump_json(),
            meta_info_json=hydra_config.meta_info.model_dump_json(),
        )


class Experiment(PolarsBaseModel):
    experiment_id: int
    name: str
    experiment_configuration_json: str

    @classmethod
    def from_raw_data(
            cls, *, assigned_id: int, experiment_name: str, hydra_config: MainConfig
    ) -> "Experiment":
        return cls(
            experiment_id=assigned_id,
            name=experiment_name,
            experiment_configuration_json=hydra_config.experiment.model_dump_json(),
        )

In [26]:
def polars_schema_to_arrow_schema(polars_schema: pl.Schema) -> pyarrow.Schema:
    return pl.DataFrame(schema=polars_schema).to_arrow().schema

In [27]:
class IDGenerator:
    _current_id: int = 0

    def get_id(self) -> int:
        _id = self._current_id
        self._current_id += 1
        return _id

    def reset(self):
        self._current_id = 0

In [28]:
def create_parquet_writer(location: Path, schema: pl.Schema) -> ParquetWriter:
    return pq.ParquetWriter(
        location,
        polars_schema_to_arrow_schema(schema),
        compression="zstd",
        use_dictionary=True,
        write_statistics=True,
    )

In [29]:
MAX_PARALLEL_REQUESTS = 3
MAX_IDS_PER_REQUEST = 200
IN_QUEUE_MAXSIZE = 1000
OUT_QUEUE_MAXSIZE = 1000
CHUNK_SIZE = 1000

QUEUE_TIMEOUT = 10  # seconds

SENTINEL = "___SENTINEL___"

logger = logging.getLogger("PARQUET BUILDER")


@dataclass(frozen=True)
class Package:
    span_id: str
    entry: TrackEntry
    config_tracking_id: int
    run_id: int

    span_info: dict[str, Any] | None = None


def main_process_loop(
        in_q: queue.Queue,
        out_q: queue.Queue,
        pool: ThreadPoolExecutor,
):
    futures: set[Future[dict[str, Any]]] = set()
    shutting_down = False

    in_flight = 0
    in_flight_lock = threading.Lock()
    all_done = threading.Event()

    def submit_batch_call(batch: list[Package]):
        nonlocal in_flight
        with in_flight_lock:
            in_flight += 1

        f = pool.submit(
            get_span_attributes,
            phoenix_graphql_endpoint=PHOENIX_ENDPOINT + "/graphql",
            spanIds=[b.span_id for b in batch],
        )
        futures.add(f)

        def _done_callback(fut: Future) -> None:
            nonlocal in_flight
            nonlocal shutting_down
            try:
                span_infos = fut.result()
                logger.warning(f"Received batch with {len(span_infos)} spans")
                for b in batch:
                    out_q.put(
                        Package(
                            run_id=b.run_id,
                            config_tracking_id=b.config_tracking_id,
                            span_id=b.span_id,
                            entry=b.entry,
                            span_info=span_infos[b.span_id],
                        )
                    )

            except (httpx.ConnectError, httpx.ReadTimeout):
                logger.error("Connection error, stopping procedure.")
                shutting_down = True
                all_done.set()

            except Exception as e:
                raise NotImplementedError()
            finally:
                futures.discard(fut)
                with in_flight_lock:
                    in_flight -= 1
                    if shutting_down and not in_flight:
                        all_done.set()

        f.add_done_callback(_done_callback)

    pending: List[Package] = []

    while True:
        while (not shutting_down) and len(pending) < MAX_IDS_PER_REQUEST:
            try:
                item = in_q.get(timeout=QUEUE_TIMEOUT)
            except queue.Empty:
                break
            except TimeoutError:
                break

            if item == SENTINEL:
                shutting_down = True
                break
            pending.append(item)

        if pending and (len(pending) >= MAX_IDS_PER_REQUEST or shutting_down):
            batch = pending[:MAX_IDS_PER_REQUEST]
            pending = pending[MAX_IDS_PER_REQUEST:]
            submit_batch_call(batch)
            continue

        if shutting_down:
            while futures:
                fut = futures.pop()
                try:
                    fut.result(timeout=QUEUE_TIMEOUT)
                except TimeoutError:
                    futures.add(fut)

            all_done.wait()
            out_q.put(SENTINEL)
            return


def drain_results_nonblocking(out_q: queue.Queue, buffer: List[Package]) -> None:
    for _ in range(MAX_IDS_PER_REQUEST):
        try:
            msg = out_q.get_nowait()
        except queue.Empty:
            return

        if msg == SENTINEL:
            out_q.put(SENTINEL)
            return

        buffer.append(msg)


async def build_parquet_files(
        runs_directory: Path,
        output_directory: Path,
):
    questions_writer = create_parquet_writer(
        output_directory / "questions.parquet", Question.get_polars_schema()
    )
    answer_writer = create_parquet_writer(
        output_directory / "answers.parquet", Answer.get_polars_schema()
    )
    run_writer = create_parquet_writer(
        output_directory / "runs.parquet", Run.get_polars_schema()
    )
    experiment_writer = create_parquet_writer(
        output_directory / "experiments.parquet", Experiment.get_polars_schema()
    )

    pool = ThreadPoolExecutor(max_workers=MAX_PARALLEL_REQUESTS)

    question_id_gen = IDGenerator()
    answer_id_gen = IDGenerator()
    run_id_gen = IDGenerator()
    experiment_id_gen = IDGenerator()
    config_tracker_id_gen = IDGenerator()

    config_tracker: dict[int, MainConfig] = {}

    buffer: list[Package] = []

    def flush(buf: list[Package]) -> None:
        logger.warning(f"flush buffer with {len(buf)} items")

        question_items = [
            Question.from_raw_data(
                assigned_id=question_id_gen.get_id(),
                entry=b.entry,
                hydra_config=config_tracker[b.config_tracking_id],
                span_attributes=b.span_info,
            )
            for b in buf
        ]

        questions_writer.write_table(Question.create_parquet_table(question_items))

        # TODO: Change the way IDs are generated when several experiments come together
        # TODO: Deduplication and ID Handling
        answer_items = [
            Answer.from_raw_data(
                assigned_id=answer_id_gen.get_id(),
                run_id=b.run_id,
                entry=b.entry,
                hydra_config=config_tracker[b.config_tracking_id],
                question_id=q.question_id,
            )
            for b, q in zip(buf, question_items)
        ]

        answer_writer.write_table(Answer.create_parquet_table(answer_items))

    in_q: queue.Queue = queue.Queue(maxsize=IN_QUEUE_MAXSIZE)
    out_q: queue.Queue = queue.Queue(maxsize=OUT_QUEUE_MAXSIZE)

    thread = threading.Thread(
        target=main_process_loop,
        args=(in_q, out_q, pool),
        daemon=True,
    )

    thread.start()

    for exp_no, experiment_name in enumerate(os.listdir(runs_directory)):
        for run_no, run_name in enumerate(
                os.listdir(runs_directory / experiment_name)
        ):
            project_path = runs_directory / experiment_name / run_name
            config_tracker[(config_id := config_tracker_id_gen.get_id())] = (
                read_hydra_config(project_path)
            )

            # TODO: correctlry handle multiple isntances of the same experiment, including ID
            experiment_id = experiment_id_gen.get_id()
            run_id = run_id_gen.get_id()

            experiment_writer.write_table(
                Experiment.create_parquet_table(
                    [
                        Experiment.from_raw_data(
                            assigned_id=experiment_id,
                            experiment_name=experiment_name,
                            hydra_config=config_tracker[config_id],
                        )
                    ]
                )
            )

            run_writer.write_table(
                Run.create_parquet_table(
                    [
                        Run.from_raw_data(
                            assigned_id=run_id,
                            hydra_config=config_tracker[config_id],
                            experiment_id=experiment_id,
                        )
                    ]
                )
            )

            for item in iter_jsonl_zst(project_path / "experiment_result.jsonl.zst"):
                tracked = TrackEntry.model_validate(item)
                in_q.put(
                    Package(
                        span_id=tracked.phoenix_span_info.span_id_hex,
                        run_id=run_id,
                        entry=tracked,
                        span_info=None,
                        config_tracking_id=config_id,
                    )
                )
                drain_results_nonblocking(out_q, buffer)

    in_q.put(SENTINEL)

    while (msg := out_q.get()) != SENTINEL:
        buffer.append(msg)
        if len(buffer) >= CHUNK_SIZE:
            flush(buffer)
            buffer.clear()

    if buffer:
        flush(buffer)

    thread.join(timeout=5)
    pool.shutdown(wait=False)
    questions_writer.close()
    answer_writer.close()

In [30]:
await build_parquet_files(Path(RUNS_DIR), Path("./results"))

Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1001 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
Received batch with 200 spans
flush buffer with 1000 items
Received batch with 200 spans
Received batch with 200 spans
Received batch 

In [31]:
Answer.get_polars_schema()

Schema([('id', Int64),
        ('run_id', Int64),
        ('message_ids', List(Int64)),
        ('question_id', Int64),
        ('phoenix_span_id', String),
        ('run_identifier', String),
        ('answers_at_beginning', List(String)),
        ('answers_at_end', List(String)),
        ('final_answer', String)])